# Value Head training (Colab L4) — Phase 3 probe

Train a tiny ValueHead on top of the **frozen** pillar2y2 backbone. The head predicts multi-horizon survival probabilities `P(survive ≥ H)` for H ∈ {25, 50, 100, 200}, masked-BCE per horizon to handle cap censoring on V11's 55% capped games.

**Architecture** (`alphatrain/value_head.py`):
```
frozen pillar2y2 backbone → (B, 256, 9, 9)
  → conv 1×1 (256 → 32) → BN → ReLU → GAP → linear (32 → 4 logits)
```
~8 K params on top of a 12 M-param frozen backbone. No policy/value gradient conflict (the failure mode of the dual-head era — HISTORY 90-118).

**Decision criterion:** if MCTS using this head beats `_2y_nb.npz` linear evaluator by ≥10% median OR materially improves P10 / %<1K at fixed sims, ship it.

**Upload to `MyDrive/alphatrain/`:**
1. `colorlines_selfplay_train.tar.gz` — code (must include the value_head + arch/value-head commits)
2. `pillar2y2_epoch_40.pt` — frozen backbone
3. `value_targets_v11.pt.gz` — ~150 MB compressed, 742 MB unpacked (built locally with build_value_targets.py)
4. `value_val_K64.pt` — 60 KB ground-truth K=64 rollout calibration set

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, time

DRIVE = '/content/drive/MyDrive/alphatrain'

!cp {DRIVE}/colorlines_selfplay_train.tar.gz /content/
!cd /content && tar xzf colorlines_selfplay_train.tar.gz

os.makedirs('/content/alphatrain/data', exist_ok=True)
shutil.copy(f'{DRIVE}/pillar2y2_epoch_40.pt', '/content/alphatrain/data/pillar2y2_epoch_40.pt')
print(f'Backbone: {os.path.getsize("/content/alphatrain/data/pillar2y2_epoch_40.pt")/1e6:.0f} MB')

shutil.copy(f'{DRIVE}/value_val_K64.pt', '/content/alphatrain/data/value_val_K64.pt')
print(f'Val set: {os.path.getsize("/content/alphatrain/data/value_val_K64.pt")/1e3:.0f} KB')

t0 = time.time()
!gzip -dc {DRIVE}/value_targets_v11.pt.gz > /content/alphatrain/data/value_targets_v11.pt
print(f'Targets: {os.path.getsize("/content/alphatrain/data/value_targets_v11.pt")/1e9:.1f} GB ({time.time()-t0:.0f}s)')

!pip install -q numpy numba scipy

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {g.total_memory / 1e9:.1f} GB')

!cd /content && python -m pytest alphatrain/tests/test_value_head.py alphatrain/tests/test_value_targets.py -q 2>&1 | tail -3

In [ ]:
# === EDIT THESE if you want a different shape ===
EPOCHS = 5
BATCH_SIZE = 4096
LR = 1e-3
HIDDEN = 32
# ==================================================

%cd /content
!python -m alphatrain.scripts.train_value_head \
    --backbone alphatrain/data/pillar2y2_epoch_40.pt \
    --train-data alphatrain/data/value_targets_v11.pt \
    --val-data alphatrain/data/value_val_K64.pt \
    --epochs {EPOCHS} --batch-size {BATCH_SIZE} --lr {LR} \
    --hidden {HIDDEN} \
    --device cuda \
    --out /content/checkpoints/value_head_v11.pt

In [ ]:
# Save trained head back to Drive
import shutil
shutil.copy('/content/checkpoints/value_head_v11.pt',
            f'{DRIVE}/value_head_v11.pt')
import os
size_kb = os.path.getsize(f'{DRIVE}/value_head_v11.pt') / 1e3
print(f'Saved value_head_v11.pt to Drive ({size_kb:.0f} KB)')

# Inspect calibration metrics from the saved checkpoint
import torch
ckpt = torch.load('/content/checkpoints/value_head_v11.pt',
                  map_location='cpu', weights_only=False)
vm = ckpt.get('val_metrics', {})
print(f'\nFinal calibration on K=64 val set (epoch {vm.get("epoch", "?")}):')
cal = vm.get('calibration', {})
for h in [25, 50, 100, 200]:
    r = cal.get(f'H{h}_r', 0)
    mae = cal.get(f'H{h}_mae', 0)
    gap = cal.get(f'H{h}_cal_gap', 0)
    print(f'  H={h:>4}: r={r:.3f}  mae={mae:.3f}  cal_gap={gap:.3f}')